uv add langchain-core langchain-community langchain-classic langchain-text-splitters \
    langchain-qdrant langchain-openai langchain-openrouter qdrant-client \
    fastembed requests \
    unstructured python-docx pymupdf pypdf

In [2]:
import os
from getpass import getpass

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key (input hidden): ")

os.environ.setdefault("OPENROUTER_MODEL", "deepseek/deepseek-v4.1-flash")

print("OpenRouter model:", os.environ["OPENROUTER_MODEL"])

OpenRouter model: deepseek/deepseek-v4.1-flash


In [3]:
import json
import math
import time
import hashlib
from collections import defaultdict
from contextlib import contextmanager
from datetime import datetime, timezone
from itertools import groupby
from pathlib import Path
from typing import Any
from uuid import NAMESPACE_URL, uuid4, uuid5

from qdrant_client import QdrantClient, models
from langchain_core.documents import Document
from langchain_core.documents.compressor import BaseDocumentCompressor
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_openrouter import ChatOpenRouter
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import (
    CSVLoader,
    PyMuPDFLoader,
    PythonLoader,
    TextLoader,
    UnstructuredHTMLLoader,
    UnstructuredWordDocumentLoader,
)

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
KNOWLEDGE_DIR = Path("knowledge")

COLLECTION_V1 = "helpdesk_production_v1"   # versioned physical collections
COLLECTION_V2 = "helpdesk_production_v2"
LIVE = "helpdesk_live"                     # alias every query goes through
TENANT_ID = "demo-tenant"
SOURCE_VERSION = "kb-v1"
PIPELINE_VERSION = "hybrid-rerank-v1"
OPENROUTER_MODEL = os.environ["OPENROUTER_MODEL"]


def print_table(rows: list[dict], columns: list[str]) -> None:
    """Plain-text table: no DataFrame needed."""
    if not rows:
        print("(no rows)")
        return
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print(" | ".join(c.ljust(widths[c]) for c in columns))
    print("-+-".join("-" * widths[c] for c in columns))
    for r in rows:
        print(" | ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))


def preview(text: str, n: int = 80) -> str:
    return " ".join(text.split())[:n]


print("Configuration loaded. Knowledge directory:", KNOWLEDGE_DIR.resolve())

c:\Users\admin\OneDrive\Desktop\AI_Engineering_Course\Week_2_Retrieval-Augmented_Generation_RAG_Foundations\AI_Eng_RAG_Week_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\admin\OneDrive\Desktop\AI_Engineering_Course\Week_2_Retrieval-Augmented_Generation_RAG_Foundations\AI_Eng_RAG_Week_2\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Configuration loaded. Knowledge directory: C:\Users\admin\OneDrive\Desktop\AI_Engineering_Course\Week_2_Retrieval-Augmented_Generation_RAG_Foundations\AI_Eng_RAG_Week_2\src\ai_eng_rag_week_2\production_RAG\knowledge


C:\Users\admin\AppData\Local\Temp\ipykernel_28752\1892319947.py:22: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [25]:
client = QdrantClient(url=QDRANT_URL, timeout=30)
print("Connected to Qdrant:", QDRANT_URL)
print("Existing collections:", [c.name for c in client.get_collections().collections])

dense_embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    openai_api_base="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")
llm = ChatOpenRouter(model=OPENROUTER_MODEL, temperature=0, max_tokens=3000, max_retries=2)

DENSE_DIM = len(dense_embeddings.embed_query("refund status for a cancelled order"))
sparse_sample = sparse_embeddings.embed_query("refund status FAQ")
print("Dense dimensions:", DENSE_DIM)
print("Sparse non-zero terms for a 3-word query:", len(sparse_sample.indices))

Connected to Qdrant: http://localhost:6333
Existing collections: ['demo_children', 'demo_faq_multivector', 'demo_runbook_contextual', 'demo_runbook_plain', 'helpdesk_production_v1']
Dense dimensions: 1536
Sparse non-zero terms for a 3-word query: 3


In [5]:
def tag(docs, **metadata):
    for doc in docs:
        doc.metadata.update(metadata)
    return docs


raw_documents = []
wiki_by_name: dict[str, list[Document]] = {}

# FAQ rows: one row = one Document; chosen columns move into metadata for filtering.
faq_docs = CSVLoader(
    file_path=str(KNOWLEDGE_DIR / "faq" / "support_faq.csv"),
    metadata_columns=["product", "category"],
).load()
raw_documents += tag(faq_docs, doc_type="faq", area="faq")
print(f"support_faq.csv       -> {len(faq_docs)} rows")

# Policy PDFs with a text layer: one Document per page.
for pdf_name in ("employee_handbook.pdf", "expense_policy.pdf"):
    pages = PyMuPDFLoader(str(KNOWLEDGE_DIR / "policies" / pdf_name)).load()
    raw_documents += tag(pages, doc_type="pdf", area="policies")
    print(f"{pdf_name:<21} -> {len(pages)} pages")

# Word memo.
memo_docs = UnstructuredWordDocumentLoader(str(KNOWLEDGE_DIR / "policies" / "travel_memo.docx")).load()
raw_documents += tag(memo_docs, doc_type="docx", area="policies")
print(f"travel_memo.docx      -> {len(memo_docs)} document(s)")

# Wiki Markdown: plain text keeps the headings for the splitter.
for md_name in ("onboarding.md", "payments_runbook.md"):
    wiki_docs = TextLoader(str(KNOWLEDGE_DIR / "wiki" / md_name), encoding="utf-8").load()
    wiki_by_name[md_name] = wiki_docs
    raw_documents += tag(wiki_docs, doc_type="markdown", area="wiki")
    print(f"{md_name:<21} -> {len(wiki_docs)} document(s)")

# Wiki HTML and Python source.
html_docs = UnstructuredHTMLLoader(str(KNOWLEDGE_DIR / "wiki" / "holiday_calendar.html")).load()
raw_documents += tag(html_docs, doc_type="html", area="wiki")
print(f"holiday_calendar.html -> {len(html_docs)} document(s)")

code_docs = PythonLoader(str(KNOWLEDGE_DIR / "wiki" / "retry_helper.py")).load()
raw_documents += tag(code_docs, doc_type="python", area="wiki")
print(f"retry_helper.py       -> {len(code_docs)} document(s)")

# Plans catalog: structured JSON turned into readable text.
plans_path = KNOWLEDGE_DIR / "catalog" / "plans.json"
plans = json.loads(plans_path.read_text())
plan_lines = [
    f"Zomato for Enterprise: {plans['billing']['cycle']} billing, "
    f"{plans['billing']['invoice']}, payment terms {plans['billing']['payment_terms_days']} days."
]
for plan in plans["plans"]:
    plan_lines.append(
        f"{plan['name']} plan: INR {plan['price_per_employee_per_month']}/employee/month, "
        f"minimum {plan['min_employees']} employees, SSO={plan['sso']}, SCIM={plan['scim']}. "
        f"Support: {plan['support']}. Features: {', '.join(plan['features'])}."
    )
for addon in plans["add_ons"]:
    plan_lines.append(f"Add-on {addon['name']}: {addon['price']} (available on {', '.join(addon['available_on'])}).")
raw_documents.append(Document(
    page_content="\n".join(plan_lines),
    metadata={"source": str(plans_path), "doc_type": "json", "area": "catalog"},
))
print("plans.json            -> 1 document")

print(f"\nTotal source documents: {len(raw_documents)}")

support_faq.csv       -> 63 rows
employee_handbook.pdf -> 8 pages
expense_policy.pdf    -> 5 pages
travel_memo.docx      -> 1 document(s)
onboarding.md         -> 1 document(s)
payments_runbook.md   -> 1 document(s)
holiday_calendar.html -> 1 document(s)
retry_helper.py       -> 1 document(s)
plans.json            -> 1 document

Total source documents: 82


## Fixed Length Chunking

In [6]:
onboarding = wiki_by_name["onboarding.md"][0]

fixed_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
fixed_chunks = fixed_splitter.split_documents([onboarding])


def show_boundaries(pieces: list[str], limit: int = 5) -> None:
    print_table(
        [
            {"#": i, "chars": len(p), "starts with": preview(p, 42), "ends with": preview(p[-42:], 42)}
            for i, p in enumerate(pieces[:limit], start=1)
        ],
        ["#", "chars", "starts with", "ends with"],
    )


print(f"Fixed-size (400 chars): {len(fixed_chunks)} chunks")
show_boundaries([c.page_content for c in fixed_chunks])

Fixed-size (400 chars): 18 chunks
# | chars | starts with                                | ends with                                 
--+-------+--------------------------------------------+-------------------------------------------
1 | 147   | # Engineering Onboarding Welcome to Zomato | oiners if anything is unclear. ## Day one 
2 | 342   | ## Day one - Collect your laptop and acces | our welcome email. ## Accounts and access 
3 | 365   | ## Accounts and access You will receive th | is reviewed every quarter. ### VPN setup  
4 | 388   | Install the company VPN client from the IT | lient settings. For anything else, contact
5 | 80    | to TCP in the client settings. For anythin | thing else, contact itdesk@zomato.example.


## Structure Aware Chunking

In [7]:
header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")],
    strip_headers=False,   # keep the heading text inside the chunk
)
sections = header_splitter.split_text(onboarding.page_content)


def section_path(metadata: dict) -> str:
    return " > ".join(metadata[h] for h in ("h1", "h2", "h3") if h in metadata)


print(f"Structure-aware: {len(sections)} chunks")
print_table(
    [{"#": i, "chars": len(s.page_content), "section": section_path(s.metadata)} for i, s in enumerate(sections[:8], start=1)],
    ["#", "chars", "section"],
)

Structure-aware: 17 chunks
# | chars | section                                                                 
--+-------+-------------------------------------------------------------------------
1 | 136   | Engineering Onboarding                                                  
2 | 319   | Engineering Onboarding > Day one                                        
3 | 351   | Engineering Onboarding > Accounts and access                            
4 | 427   | Engineering Onboarding > Accounts and access > VPN setup                
5 | 301   | Engineering Onboarding > Accounts and access > Two-factor authentication
6 | 210   | Engineering Onboarding > Accounts and access > Password manager         
7 | 370   | Engineering Onboarding > Your development environment                   
8 | 226   | Engineering Onboarding > Your development environment > Local services  


## Sementic Chunking


In [8]:
def cosine(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    return dot / ((math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b))) or 1.0)


def semantic_chunks(text: str, percentile: int = 75, min_sentences: int = 2) -> list[str]:
    sentences = [s.strip() for s in text.replace(". ", ".\n").splitlines() if s.strip()]
    vectors = dense_embeddings.embed_documents(sentences)
    distances = [1 - cosine(vectors[i], vectors[i + 1]) for i in range(len(vectors) - 1)]
    cutoff = sorted(distances)[max(0, min(len(distances) - 1, int(len(distances) * percentile / 100)))]

    chunks, current = [], [sentences[0]]
    for sentence, distance in zip(sentences[1:], distances):
        if distance > cutoff and len(current) >= min_sentences:
            chunks.append(" ".join(current))
            current = []
        current.append(sentence)
    chunks.append(" ".join(current))
    return chunks


semantic = semantic_chunks(onboarding.page_content)
print(f"Semantic: {len(semantic)} chunks")
show_boundaries(semantic)

print("\nSide by side on the same page:")
print_table(
    [
        {"strategy": name, "chunks": len(pieces), "avg chars": round(sum(map(len, pieces)) / len(pieces))}
        for name, pieces in [
            ("fixed-size (400)", [c.page_content for c in fixed_chunks]),
            ("structure-aware", [s.page_content for s in sections]),
            ("semantic (p75)", semantic),
        ]
    ],
    ["strategy", "chunks", "avg chars"],
)

Semantic: 16 chunks
# | chars | starts with                                | ends with                                 
--+-------+--------------------------------------------+-------------------------------------------
1 | 134   | # Engineering Onboarding Welcome to Zomato | sk in #new-joiners if anything is unclear.
2 | 667   | ## Day one - Collect your laptop and acces | it, and access is reviewed every quarter. 
3 | 425   | ### VPN setup Install the company VPN clie | thing else, contact itdesk@zomato.example.
4 | 508   | ### Two-factor authentication Two-factor a | xposed and tell #security-help right away.
5 | 34    | ## Your development environment 1.         | ## Your development environment 1.        

Side by side on the same page:
strategy         | chunks | avg chars
-----------------+--------+----------
fixed-size (400) | 18     | 290      
structure-aware  | 17     | 285      
semantic (p75)   | 16     | 301      


## Chunk The Whole Knowledge Base


In [9]:
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120, add_start_index=True)


def chunk_documents(docs: list[Document]) -> list[Document]:
    chunks = []
    for doc in docs:
        if doc.metadata.get("doc_type") == "markdown":
            pieces = [
                Document(page_content=s.page_content, metadata={**doc.metadata, **s.metadata, "section": section_path(s.metadata)})
                for s in header_splitter.split_text(doc.page_content)
            ]
            chunks += recursive_splitter.split_documents(pieces)
        else:
            chunks += recursive_splitter.split_documents([doc])
    return chunks


def content_hash(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8")).hexdigest()


def chunk_id(chunk: Document) -> str:
    return str(uuid5(NAMESPACE_URL, f"{chunk.metadata.get('source', 'unknown')}|{content_hash(chunk.page_content)}"))


SOURCE_PREFIXES = {
    "support_faq.csv": "FAQ", "employee_handbook.pdf": "HANDBOOK", "expense_policy.pdf": "EXPENSE",
    "travel_memo.docx": "TRAVEL", "onboarding.md": "ONBOARD",
    "payments_runbook.md": "RUNBOOK", "holiday_calendar.html": "HOLIDAY", "retry_helper.py": "CODE",
    "plans.json": "PLANS",
}
# Higher = more authoritative when two sources disagree.
AUTHORITY = {"faq": 3, "pdf": 3, "json": 3, "docx": 2, "markdown": 2, "html": 1, "python": 1}

chunks = chunk_documents(raw_documents)
citation_counters: dict[str, int] = defaultdict(int)
for chunk in chunks:
    prefix = SOURCE_PREFIXES.get(Path(chunk.metadata.get("source", "")).name, "DOC")
    citation_counters[prefix] += 1
    chunk.metadata.update(
        tenant_id=TENANT_ID,
        status="published",
        source_version=SOURCE_VERSION,
        authority=AUTHORITY.get(chunk.metadata.get("doc_type"), 1),
        content_hash=content_hash(chunk.page_content),
        citation_id=f"{prefix}-{citation_counters[prefix]:03d}",
        chunk_id=chunk_id(chunk),
    )

chunk_ids = [c.metadata["chunk_id"] for c in chunks]
ALL_CITATION_IDS = {c.metadata["citation_id"] for c in chunks}
ALLOWED_CATEGORIES = {c.metadata["category"] for c in chunks if "category" in c.metadata}
ALLOWED_PRODUCTS = {c.metadata["product"] for c in chunks if "product" in c.metadata}
ALLOWED_AREAS = {c.metadata["area"] for c in chunks}

print(f"{len(raw_documents)} source documents -> {len(chunks)} chunks")
print(f"Unique point IDs: {len(set(chunk_ids))} / {len(chunk_ids)}")
print("Chunks per source:", dict(citation_counters))
print("Categories:", sorted(ALLOWED_CATEGORIES))
print("Areas     :", sorted(ALLOWED_AREAS))
sample = next(c for c in chunks if c.metadata.get("section"))
print("\nExample structure-aware chunk:", sample.metadata["citation_id"], "|", sample.metadata["section"])
print(preview(sample.page_content, 160))

82 source documents -> 135 chunks
Unique point IDs: 135 / 135
Chunks per source: {'FAQ': 63, 'HANDBOOK': 19, 'EXPENSE': 10, 'TRAVEL': 3, 'ONBOARD': 17, 'RUNBOOK': 15, 'HOLIDAY': 2, 'CODE': 4, 'PLANS': 2}
Categories: ['Account', 'Admin', 'Billing', 'Delivery', 'Marketing', 'Membership', 'Offers', 'Onboarding', 'Operations', 'Orders', 'Payments', 'Payouts', 'Refunds', 'Safety', 'Security', 'Support']
Areas     : ['catalog', 'faq', 'policies', 'wiki']

Example structure-aware chunk: ONBOARD-001 | Engineering Onboarding
# Engineering Onboarding Welcome to Zomato Engineering. This page covers your first weeks. Ask in #new-joiners if anything is unclear.


In [10]:
PAYLOAD_INDEX_FIELDS = ["tenant_id", "status", "category", "product", "doc_type", "area", "source"]

def prepare_collection(name: str) -> None:
    if not client.collection_exists(name):
        client.create_collection(
            collection_name=name,
            vectors_config={"": models.VectorParams(size=DENSE_DIM, distance=models.Distance.COSINE)},
            sparse_vectors_config={"langchain-sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)},
            hnsw_config=models.HnswConfigDiff(m=16, ef_construct=128),
        )
    for field in PAYLOAD_INDEX_FIELDS:
        try:
            client.create_payload_index(
                collection_name=name,
                field_name=f"metadata.{field}",   # LangChain nests metadata under "metadata"
                field_schema=models.PayloadSchemaType.KEYWORD,
            )
        except Exception as exc:   # some versions complain if the index already exists
            print(f"Payload index note for {field}: {exc}")


def build_store(collection: str, mode: RetrievalMode = RetrievalMode.HYBRID) -> QdrantVectorStore:
    return QdrantVectorStore(
        client=client,
        collection_name=collection,
        retrieval_mode=mode,
        embedding=None if mode == RetrievalMode.SPARSE else dense_embeddings,
        sparse_embedding=None if mode == RetrievalMode.DENSE else sparse_embeddings,
    )


def index_chunks(collection: str, docs: list[Document]) -> None:
    prepare_collection(collection)
    build_store(collection).add_documents(docs, ids=[d.metadata["chunk_id"] for d in docs])


def switch_alias(alias: str, target: str) -> None:
    """Atomically point an alias at a collection (creating it if needed)."""
    ops = []
    if alias in {a.alias_name for a in client.get_aliases().aliases}:
        ops.append(models.DeleteAliasOperation(delete_alias=models.DeleteAlias(alias_name=alias)))
    ops.append(models.CreateAliasOperation(create_alias=models.CreateAlias(collection_name=target, alias_name=alias)))
    client.update_collection_aliases(change_aliases_operations=ops)


index_chunks(COLLECTION_V1, chunks)   # idempotent: stable IDs mean upsert, not duplicate
switch_alias(LIVE, COLLECTION_V1)

print("Physical collection:", COLLECTION_V1, "-> alias:", LIVE)
print("Points:", client.count(LIVE, exact=True).count)

Physical collection: helpdesk_production_v1 -> alias: helpdesk_live
Points: 135


## Metadata Filters


In [11]:
def access_filter(tenant_id: str, **equals: str | None) -> models.Filter:
    must = [
        models.FieldCondition(key="metadata.tenant_id", match=models.MatchValue(value=tenant_id)),
        models.FieldCondition(key="metadata.status", match=models.MatchValue(value="published")),
    ]
    for field, value in equals.items():
        if value is not None:
            must.append(models.FieldCondition(key=f"metadata.{field}", match=models.MatchValue(value=value)))
    return models.Filter(must=must)


ALLOWLISTS = {"category": ALLOWED_CATEGORIES, "product": ALLOWED_PRODUCTS, "area": ALLOWED_AREAS}


def validated_filter(tenant_id: str, **business: str | None) -> models.Filter:
    for field, value in business.items():
        if value is not None and value not in ALLOWLISTS[field]:
            raise ValueError(f"Unsupported {field}: {value!r}")
    return access_filter(tenant_id, **business)


ACCESS_FILTER = access_filter(TENANT_ID)
print("Access filter ready:", [c.key for c in ACCESS_FILTER.must])

Access filter ready: ['metadata.tenant_id', 'metadata.status']


## Retrieval : Dense and Sparse and Hybrid

In [12]:
dense_store = build_store(LIVE, RetrievalMode.DENSE)
sparse_store = build_store(LIVE, RetrievalMode.SPARSE)
hybrid_store = build_store(LIVE, RetrievalMode.HYBRID)


def show_results(store, query: str, k: int = 5, **kwargs):
    results = store.similarity_search_with_score(query, k=k, filter=ACCESS_FILTER, **kwargs)
    print_table(
        [
            {"rank": r, "id": d.metadata["citation_id"], "score": round(s, 3), "content": preview(d.page_content, 700)}
            for r, (d, s) in enumerate(results, start=1)
        ],
        ["rank", "id", "score", "content"],
    )
    return [d for d, _ in results]


for query in ("My money has not come back after the order was cancelled", "Does the plan support SCIM provisioning?"):
    print(f"\n=== {query}")
    for name, store in (("dense", dense_store), ("sparse / BM25", sparse_store), ("hybrid (server-side RRF)", hybrid_store)):
        print(f"\n{name}")
        show_results(store, query, k=3)


=== My money has not come back after the order was cancelled

dense
rank | id      | score | content                                                                                                                                                                                              
-----+---------+-------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
1    | FAQ-001 | 0.539 | question: How do I get a refund for a cancelled order? answer: Refunds for cancelled orders go back to the original payment method within 5 to 7 working days. Zomato credits are refunded instantly.
2    | FAQ-057 | 0.497 | question: My payment failed but money was deducted. What now? answer: In most cases the amount is automatically reversed within 5 to 7 working days. If not, contact support with the transaction ID.
3    | FAQ-044 | 0.478 | question: How 

## Metadata Filtering

In [13]:
question = "Why was the amount lower than I expected?"

print("No business filter")
show_results(hybrid_store, question, k=3)

print("\ncategory = Payouts (validated against the allowlist)")
payout_docs = hybrid_store.similarity_search_with_score(question, k=3, filter=validated_filter(TENANT_ID, category="Payouts"))
print_table(
    [{"id": d.metadata["citation_id"], "category": d.metadata["category"], "content": preview(d.page_content, 700)} for d, _ in payout_docs],
    ["id", "category", "content"],
)

try:
    validated_filter(TENANT_ID, category="Payouts'; DROP")
except ValueError as exc:
    print("\nRejected unknown category:", exc)

No business filter
rank | id           | score | content                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     
-----+--------------+-------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Contextual Retreival

In [14]:
runbook = wiki_by_name["payments_runbook.md"][0]
runbook_chunks = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=50).split_documents([runbook])

contextual_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Write one or two sentences that situate a chunk within its source document so it can be found by search. "
     "Name the topic, system or procedure the chunk belongs to. Use only the document. Return only the context."),
    ("human", "<document>\n{document}\n</document>\n\n<chunk>\n{chunk}\n</chunk>"),
])
contextual_chain = contextual_prompt | llm | StrOutputParser()

contexts = contextual_chain.batch(
    [{"document": runbook.page_content, "chunk": c.page_content} for c in runbook_chunks],
    config={"max_concurrency": 4},
)
contextual_chunks = [
    Document(
        page_content=f"{ctx.strip()}\n\n{c.page_content}",
        metadata={**c.metadata, "original_text": c.page_content, "chunk_id": str(uuid5(NAMESPACE_URL, f"ctx|{i}"))},
    )
    for i, (c, ctx) in enumerate(zip(runbook_chunks, contexts))
]
print(f"{len(contextual_chunks)} chunks contextualised. Example:")
print(" context:", preview(contexts[-3], 800))

16 chunks contextualised. Example:
 context: This chunk is from the Payments Service Runbook, in the sections on Reconciliation and Escalation. It covers the daily reconciliation procedure where Finance compares gateway settlement files against internal records, alerts #payments-recon for mismatches above ₹10,000, and the on-call engineer exports payment events for the affected window, followed by the start of the Escalation section.


In [15]:
def demo_store(name: str, docs: list[Document]) -> QdrantVectorStore:
    """Throwaway hybrid collection for a side-by-side experiment."""
    if client.collection_exists(name):
        client.delete_collection(name)
    prepare_collection(name)
    store = build_store(name)
    store.add_documents(docs, ids=[str(uuid5(NAMESPACE_URL, f"{name}|{i}")) for i in range(len(docs))])
    return store


plain_store = demo_store("demo_runbook_plain", runbook_chunks)
contextual_store = demo_store("demo_runbook_contextual", contextual_chunks)

In [16]:
TARGET = "Never restart all regions at once"   # the chunk that answers the questions below; it never says "payments"


def rank_of_target(store: QdrantVectorStore, query: str) -> int | str:
    results = store.similarity_search(query, k=len(runbook_chunks))
    return next((r for r, d in enumerate(results, 1) if TARGET in d.metadata.get("original_text", d.page_content)), "-")


queries = [
    "What are the restart prohibitions for the payments service?",
    "Which mistakes should I avoid when bouncing the payments service?",
    "Any payments service restart safety rules besides the numbered steps?",
]
print_table(
    [{"query": q, "rank (plain)": rank_of_target(plain_store, q), "rank (contextual)": rank_of_target(contextual_store, q)} for q in queries],
    ["query", "rank (plain)", "rank (contextual)"],
)

query                                                                 | rank (plain) | rank (contextual)
----------------------------------------------------------------------+--------------+------------------
What are the restart prohibitions for the payments service?           | 5            | 1                
Which mistakes should I avoid when bouncing the payments service?     | 11           | 1                
Any payments service restart safety rules besides the numbered steps? | 7            | 3                


## Parent Document Retrieval

In [17]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore

child_store = demo_store("demo_children", [])   # empty hybrid collection

parent_child = ParentDocumentRetriever(
    vectorstore=child_store,
    docstore=InMemoryStore(),
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30),
    parent_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0),
    search_kwargs={"k": 4},
)
parent_child.add_documents(wiki_by_name["payments_runbook.md"] + wiki_by_name["onboarding.md"])

query = "Which command do I run to roll back the payments deployment?"
child_hit = child_store.similarity_search(query, k=4)[0]   # same search the retriever runs
parent_hit = parent_child.invoke(query)[0]

print(f"Matched child  ({len(child_hit.page_content)} chars):\n  {preview(child_hit.page_content, 200)}")
print(f"\nReturned parent ({len(parent_hit.page_content)} chars):\n  {preview(parent_hit.page_content, 400)}")

Matched child  (287 chars):
  Deploy through the pipeline during the weekday window. To roll back, run `kubectl rollout undo deployment/payments -n payments-prod` and confirm the version with `kubectl rollout history`. Database mi

Returned parent (819 chars):
  Deploy through the pipeline during the weekday window. To roll back, run `kubectl rollout undo deployment/payments -n payments-prod` and confirm the version with `kubectl rollout history`. Database migrations must be backward compatible for one release so that a rollback is always safe. ## Common incidents ### A single bank is failing If success drops for one bank only, check the gateway status pa


## Multi-vector Retrieval

In [18]:
MV_COLLECTION = "demo_faq_multivector"
faq_chunks = [c for c in chunks if c.metadata.get("doc_type") == "faq"]
faq_questions = [c.page_content.split("\nanswer: ")[0].removeprefix("question: ") for c in faq_chunks]
faq_answers = [c.page_content.split("\nanswer: ", 1)[1] for c in faq_chunks]

if client.collection_exists(MV_COLLECTION):
    client.delete_collection(MV_COLLECTION)
client.create_collection(
    MV_COLLECTION,
    vectors_config={
        "question": models.VectorParams(size=DENSE_DIM, distance=models.Distance.COSINE),
        "answer": models.VectorParams(size=DENSE_DIM, distance=models.Distance.COSINE),
    },
)
client.upsert(
    MV_COLLECTION,
    points=[
        models.PointStruct(id=c.metadata["chunk_id"], vector={"question": q, "answer": a},
                           payload={"citation_id": c.metadata["citation_id"], "text": preview(c.page_content, 120)})
        for c, q, a in zip(
            faq_chunks,
            dense_embeddings.embed_documents(faq_questions),
            dense_embeddings.embed_documents(faq_answers),
        )
    ],
)


def multivector_search(query: str, using: str | None, k: int = 3) -> list[str]:
    vector = dense_embeddings.embed_query(query)
    if using is None:   # fuse both named vectors with RRF
        response = client.query_points(
            MV_COLLECTION,
            prefetch=[
                    models.Prefetch(query=vector, using="question", limit=10),
                    models.Prefetch(query=vector, using="answer", limit=10)],
            query=models.FusionQuery(fusion=models.Fusion.RRF), limit=k, with_payload=True,
        )
    else:
        response = client.query_points(MV_COLLECTION, query=vector, using=using, limit=k, with_payload=True)
    return [p.payload["citation_id"] for p in response.points]


for query in ("Can I get my money back if I cancel?", "Refunds go to the original payment method in 5 to 7 working days"):
    print(f"\n{query}")
    print_table(
        [{"search": label, "top-3 ids": ", ".join(multivector_search(query, using))} for label, using in
         (("question vector", "question"), ("answer vector", "answer"), ("fused (RRF)", None))],
        ["search", "top-3 ids"],
    )


Can I get my money back if I cancel?
search          | top-3 ids                
----------------+--------------------------
question vector | FAQ-001, FAQ-011, FAQ-044
answer vector   | FAQ-011, FAQ-001, FAQ-052
fused (RRF)     | FAQ-001, FAQ-011, FAQ-003

Refunds go to the original payment method in 5 to 7 working days
search          | top-3 ids                
----------------+--------------------------
question vector | FAQ-002, FAQ-001, FAQ-004
answer vector   | FAQ-001, FAQ-002, FAQ-057
fused (RRF)     | FAQ-001, FAQ-002, FAQ-004


# Query Intelligence

In [19]:
def compare_rankings(label_a: str, docs_a: list[Document], label_b: str, docs_b: list[Document]) -> None:
    rows = []
    for rank in range(max(len(docs_a), len(docs_b))):
        a = docs_a[rank] if rank < len(docs_a) else None
        b = docs_b[rank] if rank < len(docs_b) else None
        rows.append({
            "rank": rank + 1,
            label_a: f"{a.metadata['citation_id']} {preview(a.page_content, 80)}" if a else "",
            label_b: f"{b.metadata['citation_id']} {preview(b.page_content, 80)}" if b else "",
        })
    print_table(rows, ["rank", label_a, label_b])

## Query Rewriting

In [20]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Rewrite a user question into one concise standalone search query. "
     "Preserve exact product names, plan names, IDs, dates, and negation. "
     "Do not answer the question. Return only the rewritten query."),
    ("human", "{question}"),
])
rewrite_chain = rewrite_prompt | llm | StrOutputParser()


def rewrite_query(question: str) -> str:
    return rewrite_chain.invoke({"question": question}).strip().strip('"')


question = "I paid for the Business plan but we still don't have SSO. Why?"
rewritten = rewrite_query(question)
print("Original :", question)
print("Rewritten:", rewritten)

must_keep = ["Business", "SSO"]
print("Details preserved:", {term: term.lower() in rewritten.lower() for term in must_keep})
compare_rankings("original", hybrid_store.similarity_search(question, k=3, filter=ACCESS_FILTER),
                 "rewritten", hybrid_store.similarity_search(rewritten, k=3, filter=ACCESS_FILTER))

Original : I paid for the Business plan but we still don't have SSO. Why?
Rewritten: Business plan paid but SSO still not available
Details preserved: {'Business': True, 'SSO': True}
rank | original                                                                                   | rewritten                                                                                 
-----+--------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------
1    | FAQ-025 question: Which plan includes single sign-on? answer: SSO is available on the Bu   | FAQ-025 question: Which plan includes single sign-on? answer: SSO is available on the Bu  
2    | FAQ-026 question: Which identity providers are supported for SSO? answer: SAML 2.0 provi   | PLANS-001 Zomato for Enterprise: monthly billing, one consolidated GST invoice, payment te
3    | PLANS-001 Zomato for Enterprise: monthly billi

## Multi-query retrieval

In [21]:
def rrf(rankings: list[list[Document]], k: int = 60) -> list[tuple[Document, float]]:
    """Fuse ranked lists by chunk_id: score = sum(1 / (k + rank))."""
    scores: dict[str, float] = defaultdict(float)
    docs: dict[str, Document] = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            docs[doc.metadata["chunk_id"]] = doc
            scores[doc.metadata["chunk_id"]] += 1.0 / (k + rank)
    return [(docs[cid], score) for cid, score in sorted(scores.items(), key=lambda item: item[1], reverse=True)]

In [22]:
multi_query_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Generate exactly 3 different search queries that retrieve evidence for the user's question. "
     "Preserve entities and constraints. Return one query per line, no numbering."),
    ("human", "{question}"),
])
multi_query_chain = multi_query_prompt | llm | StrOutputParser()


def multi_query_search(question: str, k_per_query: int = 5, final_k: int = 6):
    raw = multi_query_chain.invoke({"question": question})
    variants = list(dict.fromkeys(line.strip(" -*\t") for line in raw.splitlines() if line.strip()))[:3]
    queries = [question] + [q for q in variants if q.casefold() != question.casefold()]
    rankings = [hybrid_store.similarity_search(q, k=k_per_query, filter=ACCESS_FILTER) for q in queries]
    return queries[1:], [doc for doc, _ in rrf(rankings)[:final_k]]


generated, merged = multi_query_search("Can our company sync employee accounts automatically and get a dedicated account manager?")
print("Generated queries:", *generated, sep="\n - ")
print()
print_table(
    [{"rank": r, "id": d.metadata["citation_id"], "content": preview(d.page_content, 80)} for r, d in enumerate(merged, 1)],
    ["rank", "id", "content"],
)

Generated queries:
 - "sync employee accounts automatically" "dedicated account manager"
 - "automatic employee account provisioning" "dedicated account manager

rank | id          | content                                                                         
-----+-------------+---------------------------------------------------------------------------------
1    | PLANS-002   | Enterprise plan: INR 349/employee/month, minimum 500 employees, SSO=True, SCIM=T
2    | FAQ-031     | question: How do I add or remove employees in bulk? answer: Upload a CSV with na
3    | FAQ-027     | question: Does Zomato for Enterprise support SCIM user provisioning? answer: Aut
4    | ONBOARD-003 | ## Accounts and access You will receive these accounts on day one: company email
5    | FAQ-033     | question: What support does each Enterprise plan include? answer: Starter has em
6    | FAQ-062     | question: How do I contact customer support? answer: Use Help in the app for cha


## Hyde

In [23]:
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Write a short, plausible help-center answer (2 to 3 sentences) to the question. "
     "It is only used as a search probe, so do not add disclaimers."),
    ("human", "{question}"),
])
hyde_chain = hyde_prompt | llm | StrOutputParser()

question = "Why can't I get my money back after cancelling?"
hypothetical = hyde_chain.invoke({"question": question})
print("Hypothetical answer:", preview(hypothetical, 220), "\n")
compare_rankings("direct query", dense_store.similarity_search(question, k=3, filter=ACCESS_FILTER),
                 "HyDE probe", dense_store.similarity_search(hypothetical, k=3, filter=ACCESS_FILTER))

Hypothetical answer: You may not receive a refund because your cancellation fell outside the eligible refund window, or the charge was for a non-refundable item, service, or fee. Some accounts also deduct usage, setup, or processing costs be 

rank | direct query                                                                             | HyDE probe                                                                              
-----+------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------
1    | FAQ-001 question: How do I get a refund for a cancelled order? answer: Refunds for cance | FAQ-004 question: I was refunded less than I paid. Why? answer: Delivery tips and certai
2    | FAQ-057 question: My payment failed but money was deducted. What now? answer: In most ca | FAQ-001 question: How do I get a refund for a cancelled order? answer: Refunds for cance
3    | F

## Self Query - Structured Output

In [26]:
self_query_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Split the user's question into a search query and optional metadata filters. "
     "Allowed categories: {categories}. Allowed products: {products}. "
     'Return JSON only: {{"query": str, "category": str or null, "product": str or null}}. '
     "Use null unless the question clearly names one."),
    ("human", "{question}"),
])
self_query_chain = self_query_prompt | llm | JsonOutputParser()


def self_query_search(question: str, k: int = 3):
    parsed = self_query_chain.invoke({
        "question": question,
        "categories": ", ".join(sorted(ALLOWED_CATEGORIES)),
        "products": ", ".join(sorted(ALLOWED_PRODUCTS)),
    })
    proposed = {"category": parsed.get("category"), "product": parsed.get("product")}
    accepted = {f: v for f, v in proposed.items() if v in ALLOWLISTS[f]}   # unknown values are dropped
    docs = hybrid_store.similarity_search(parsed.get("query") or question, k=k, filter=access_filter(TENANT_ID, **accepted))
    return proposed, accepted, docs


proposed, accepted, docs = self_query_search("How do restaurant partners dispute a cancellation charge?")
print("LLM proposed :", proposed)
print("Filters used :", accepted)
print_table(
    [{"id": d.metadata["citation_id"], "category": d.metadata.get("category", "-"), "content": preview(d.page_content, 70)} for d in docs],
    ["id", "category", "content"],
)

LLM proposed : {'category': 'Payments', 'product': 'Restaurant Partners'}
Filters used : {'category': 'Payments', 'product': 'Restaurant Partners'}
(no rows)


## Query Routing

In [ ]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Choose where to search. Areas: faq (customer, partner and enterprise help-center answers), "
     "policies (employee handbook, expenses, travel), wiki (engineering onboarding, payments runbook, holiday calendar, code), "
     "catalog (enterprise plans and prices). "
     'Return JSON only: {{"area": "<one area or all>"}}.'),
    ("human", "{question}"),
])
router_chain = router_prompt | llm | JsonOutputParser()


def route_area(question: str) -> str | None:
    area = router_chain.invoke({"question": question}).get("area")
    return area if area in ALLOWED_AREAS else None   # None = search every area


for question in ("How do I roll back a payments deployment?", "What does the Business plan cost per employee?", "How long does a refund take?"):
    area = route_area(question)
    top = hybrid_store.similarity_search(question, k=1, filter=validated_filter(TENANT_ID, area=area))[0]
    print(f"{question:<50} -> area={area!s:<8} top hit: {top.metadata['citation_id']}")

How do I roll back a payments deployment?          -> area=wiki     top hit: RUNBOOK-006
What does the Business plan cost per employee?     -> area=catalog  top hit: PLANS-002
How long does a refund take?                       -> area=faq      top hit: FAQ-002
